## Daily readiness dashboard
#### Loads objective and subjective wellness data alongside recent activity data to suggest training priorities for the day

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from hll.config import PROCESSED_DIR

# Day to show. None means "latest day that has data". Set e.g. "2026-10-18" to look back.
SELECTED_DATE = None

In [ ]:
# Colours
BG, CARD = "#12141f", "#1a1d2e"
INK, MUTED, GRID = "#e8eaf2", "#8b90a7", "#2a2e45"
GOOD, WARN, CRIT = "#0ca30c", "#fab219", "#d03b3b"
BLUE, ORANGE = "#3987e5", "#d95926"

# Each status label has a colour and an icon, so colour is never the only signal.
STATUS = {
    "OK": (GOOD, "●"),
    "Watch": (WARN, "▲"),
    "Flag": (CRIT, "✖"),
    "No data": (MUTED, "–"),
}

# Thresholds: my starting heuristics, not validated cut-offs.
HRV_WATCH_SD, HRV_FLAG_SD = 0.5, 1.0       # SDs below baseline
CV_WATCH, CV_FLAG = 1.25, 1.5              # multiples of your typical CV
RHR_WATCH_BPM, RHR_FLAG_BPM = 2, 4         # bpm above baseline
SLEEP_WATCH_H, SLEEP_FLAG_H = 0.75, 1.5    # hours below your 7-night average
AC_LOW, AC_HIGH, AC_FLAG = 0.8, 1.3, 1.5   # acute:chronic bands

In [ ]:
def show(value, decimals=0, sign=False):
    """Number to text; sign=True adds a + for positive numbers. Missing -> dash."""
    if pd.isna(value):
        return "–"
    plus = "+" if sign else ""
    return f"{value:{plus}.{decimals}f}"


def window(series, end_day, n_days):
    """The last n_days of a series, up to and including end_day."""
    return series.loc[end_day - pd.Timedelta(days=n_days - 1): end_day]


def rate(badness, watch, flag):
    """Bigger badness = worse. Returns 'OK', 'Watch', 'Flag' or 'No data'."""
    if pd.isna(badness):
        return "No data"
    if badness >= flag:
        return "Flag"
    if badness >= watch:
        return "Watch"
    return "OK"


def rate_ratio(r):
    """Acute:chronic. 'Watch' can mean too high or too low; the detail line shows which."""
    if pd.isna(r):
        return "No data"
    if r >= AC_FLAG:
        return "Flag"
    if r > AC_HIGH or r < AC_LOW:
        return "Watch"
    return "OK"


# Styles for the tiles, named once so the tile function stays readable.
TILE = f"flex:1; background:{CARD}; border-radius:8px; padding:16px 14px;"
LABEL = f"color:{MUTED}; font-size:13px;"
VALUE = f"color:{INK}; font-size:32px; font-weight:700;"
STATUS_TEXT = f"color:{INK}; font-size:13px; font-weight:600; margin-top:8px;"


def status_tile(label, value, unit, detail, status):
    """One tile: label, big number, detail line, and a coloured edge with icon and status word."""
    colour, icon = STATUS[status]
    return f"""
    <div style="{TILE} border-left:6px solid {colour};">
      <div style="{LABEL}">{label}</div>
      <div style="{VALUE}">{value} <span style="{LABEL}">{unit}</span></div>
      <div style="{LABEL}">{detail}</div>
      <div style="{STATUS_TEXT}"><span style="color:{colour};">{icon}</span> {status}</div>
    </div>"""


def tile_row(tiles_html):
    """Put tiles side by side."""
    return f'<div style="display:flex; gap:12px; margin-bottom:12px;">{tiles_html}</div>'

In [ ]:
well = pd.read_csv(PROCESSED_DIR / "wellness.csv", parse_dates=["date"])
well = well.sort_values("date").set_index("date")

print("rows", len(well), "| range", well.index.min().date(), "to", well.index.max().date())

well.isna().mean().round(2)

In [ ]:
# Add a row for any missing calendar day, so a "7-day" window always means 7 days.
well = well.asfreq("D")
well["sleep_h"] = well["sleep_secs"] / 3600

In [ ]:
# last_valid_index() is the date of the last row with an HRV value, so if today's
# data hasn't synced yet we show the most recent day that has it.
if SELECTED_DATE is None:
    day = well["hrv"].last_valid_index()
else:
    day = pd.Timestamp(SELECTED_DATE)

print("Showing:", day.date())

In [ ]:
hrv_last7 = window(well["hrv"], day, 7)
hrv_last60 = window(well["hrv"], day, 60)

hrv_recent = hrv_last7.mean()              # recent HRV
hrv_baseline = hrv_last60.mean()           # your normal
hrv_baseline_sd = hrv_last60.std()         # how much it normally varies
hrv_cv = 100 * hrv_last7.std() / hrv_last7.mean()

# Typical CV: the 7-day CV worked out for every day, averaged over the last 60 days.
cv_by_day = 100 * well["hrv"].rolling(7).std() / well["hrv"].rolling(7).mean()
hrv_cv_baseline = window(cv_by_day, day, 60).mean()

rhr_recent = window(well["resting_hr"], day, 7).mean()
rhr_baseline = window(well["resting_hr"], day, 60).mean()

sleep_last_night = well.loc[day, "sleep_h"]
sleep_usual = window(well["sleep_h"], day - pd.Timedelta(days=1), 7).mean()   # the 7 nights before

In [ ]:
hrv_pct = 100 * (hrv_recent / hrv_baseline - 1)          # % vs baseline
hrv_z = (hrv_recent - hrv_baseline) / hrv_baseline_sd    # distance from baseline in SDs
cv_ratio = hrv_cv / hrv_cv_baseline                      # 1.0 = typical
rhr_diff = rhr_recent - rhr_baseline                     # bpm vs baseline
sleep_shortfall = sleep_usual - sleep_last_night         # hours less than usual

hrv_status = rate(-hrv_z, HRV_WATCH_SD, HRV_FLAG_SD)     # only drops in HRV are flagged
cv_status = rate(cv_ratio, CV_WATCH, CV_FLAG)
rhr_status = rate(rhr_diff, RHR_WATCH_BPM, RHR_FLAG_BPM)
sleep_status = rate(sleep_shortfall, SLEEP_WATCH_H, SLEEP_FLAG_H)

In [ ]:
act = pd.read_csv(PROCESSED_DIR / "activities.csv", parse_dates=["start_date"])
act["date"] = act["start_date"].dt.normalize()

daily_load = act.groupby("date")["icu_load"].sum()      # total load per day

# Rest days have no activities, so add them back with zero load. The window stops at the
# day BEFORE the selected morning, because that day's training hasn't happened yet.
load_end = day - pd.Timedelta(days=1)
all_days = pd.date_range(daily_load.index.min(), load_end)
daily_load = daily_load.reindex(all_days, fill_value=0)

In [ ]:
acute = daily_load.rolling(7).mean()       # average daily load, last 7 days
chronic = daily_load.rolling(28).mean()    # average daily load, last 28 days

acute_now = acute.iloc[-1]
chronic_now = chronic.iloc[-1]
ac_ratio = acute_now / chronic_now
ac_status = rate_ratio(ac_ratio)

print("load up to", load_end.date(), "| acute", show(acute_now), "| chronic", show(chronic_now),
      "| ratio", show(ac_ratio, 2))

In [ ]:
readiness_tiles = (
    status_tile("HRV (rMSSD), 7-day mean", show(hrv_recent), "ms",
                f"{show(hrv_pct, 1, sign=True)}% vs 60-day baseline ({show(hrv_baseline)} ms)", hrv_status)
    + status_tile("HRV variability, 7-day CV", show(hrv_cv, 1), "%",
                  f"typical {show(hrv_cv_baseline, 1)}% ({show(cv_ratio, 2)}×)", cv_status)
    + status_tile("Resting HR, 7-day mean", show(rhr_recent, 1), "bpm",
                  f"{show(rhr_diff, 1, sign=True)} bpm vs baseline ({show(rhr_baseline, 1)})", rhr_status)
)

sleep_tile = status_tile("Sleep, last night", show(sleep_last_night, 1), "h",
                         f"7-night average {show(sleep_usual, 1)} h", sleep_status)

load_tile = status_tile("Acute:chronic load", show(ac_ratio, 2), "",
                        f"acute {show(acute_now)} · chronic {show(chronic_now)} (avg daily load)", ac_status)

In [ ]:
recent = pd.DataFrame({"acute": acute, "chronic": chronic}).tail(56)   # last 8 weeks

import plotly.graph_objects as go

fig = go.Figure()
fig.add_scatter(x=recent.index, y=recent["acute"], name="Acute (7-day avg)",
                line=dict(color=BLUE, width=2))
fig.add_scatter(x=recent.index, y=recent["chronic"], name="Chronic (28-day avg)",
                line=dict(color=ORANGE, width=2))
fig.update_layout(
    template="plotly_dark", height=320,
    paper_bgcolor=CARD, plot_bgcolor=CARD,
    title=dict(text="Acute vs chronic load", font=dict(size=16, color=INK)),
    legend=dict(orientation="h", y=1.15, x=0),
    margin=dict(t=70, l=50, r=20, b=40),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor=GRID, title_text="Avg daily load")

In [ ]:
display(HTML(f"""
<div style="background:{BG}; padding:16px 16px 4px; border-radius:10px; font-family:sans-serif;">
  <div style="color:{INK}; font-size:18px; margin-bottom:12px;">Readiness · {day:%a %d %b %Y}</div>
  {tile_row(readiness_tiles)}
  {tile_row(sleep_tile + load_tile)}
</div>"""))

fig.show()